In [ ]:
# Importando as Bibliotecas
import pandas as pd
from pandas_gbq import to_gbq
from google.cloud import storage
from email import message_from_string
import logging

In [ ]:
# Configurando logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Inicializando as variáveis
nome_do_bucket_eml = 'project-b2d9e7e0-e964-49fe-935-emails-datalake'
projeto_bigquery = 'project-b2d9e7e0-e964-49fe-935'
dataset_bigquery = 'insight_data'
tabela_bigquery = f'{dataset_bigquery}.tb_emails_feedback'

In [ ]:
# Inicializando o cliente de armazenamento do Google Cloud
try:
    storage_client = storage.Client()
    logging.info("Cliente de armazenamento do Google Cloud inicializado com sucesso.")
except Exception as e:
    logging.error(f"Falha ao inicializar o cliente de armazenamento do Google Cloud. Erro: {e}")
    raise e

In [ ]:
# Listando todos os blobs no bucket    
try:
    blobs = list(storage_client.list_blobs(nome_do_bucket_eml))
    logging.info(f"Listando os blobs do bucket {nome_do_bucket_eml}...")    
except Exception as e:
    logging.error(f"Falha ao listar os blobs. Erro: {e}")
    raise e

In [ ]:
blobs

In [7]:
pastas = list(set([blob.name.split('/')[0] for blob in blobs if '/' in blob.name]))

In [8]:
pastas

['2026-05-04-22-32', '2026-05-04-21-59']

In [12]:
# Filtrar as pastas que seguem o formato de data e hora (yyyy-mm-dd-hh-mm)
try: 
    pastas = list(set([blob.name.split('/')[0] for blob in blobs if '/' in blob.name]))
    pastas_ordenadas = sorted(pastas, reverse=True)
    ultima_pasta = pastas_ordenadas[0]
    pasta_eml = f'{ultima_pasta}/emails_feedback_faker/'
    logging.info(f"Última pasta de data encontrada: {pasta_eml}")

    blobs_eml = list(storage_client.list_blobs(nome_do_bucket_eml, prefix=pasta_eml))
    arquivos_eml = [blob.name for blob in blobs_eml if blob.name.endswith('.eml')]
    logging.info(f"Arquivos .eml encontrados: {arquivos_eml}")
except Exception as e:
    logging.error(f"Falha ao listar as pastas e arquivos. Erro: {e}")
    raise e

2026-05-08 01:24:14,114 - INFO - Última pasta de data encontrada: 2026-05-04-22-32/emails_feedback_faker/
2026-05-08 01:24:14,242 - INFO - Arquivos .eml encontrados: ['2026-05-04-22-32/emails_feedback_faker/email_feedback_1.eml', '2026-05-04-22-32/emails_feedback_faker/email_feedback_10.eml', '2026-05-04-22-32/emails_feedback_faker/email_feedback_100.eml', '2026-05-04-22-32/emails_feedback_faker/email_feedback_101.eml', '2026-05-04-22-32/emails_feedback_faker/email_feedback_102.eml', '2026-05-04-22-32/emails_feedback_faker/email_feedback_103.eml', '2026-05-04-22-32/emails_feedback_faker/email_feedback_104.eml', '2026-05-04-22-32/emails_feedback_faker/email_feedback_105.eml', '2026-05-04-22-32/emails_feedback_faker/email_feedback_106.eml', '2026-05-04-22-32/emails_feedback_faker/email_feedback_107.eml', '2026-05-04-22-32/emails_feedback_faker/email_feedback_108.eml', '2026-05-04-22-32/emails_feedback_faker/email_feedback_109.eml', '2026-05-04-22-32/emails_feedback_faker/email_feedback_1

In [19]:
# Função para extrair informações dos e-mails
def extrair_informacoes_eml(conteudo_eml):
    mensagem = message_from_string(conteudo_eml)
    remetente = mensagem.get('From')
    destinatario = mensagem.get('To')
    assunto = mensagem.get('Subject')
    data = mensagem.get('Date')
    corpo = mensagem.get_payload()
    return {
        'remetente': remetente,
        'destinatario': destinatario,
        'assunto': assunto,
        'data': data,
        'corpo': corpo
    }

In [20]:
dados_emails = []

for arquivo in arquivos_eml:
    try:
        logging.info(f"Lendo arquivo .eml: {arquivo}")
        blob = storage_client.bucket(nome_do_bucket_eml).blob(arquivo)
        conteudo_eml = blob.download_as_text()
        
        dados_extraidos = extrair_informacoes_eml(conteudo_eml)
        dados_emails.append(dados_extraidos)
    except Exception as e:
        logging.error(f"Erro ao processar o arquivo {arquivo}. Erro: {e}")
        continue

2026-05-08 01:38:57,736 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_1.eml
2026-05-08 01:38:57,868 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_10.eml
2026-05-08 01:38:57,937 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_100.eml
2026-05-08 01:38:58,008 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_101.eml
2026-05-08 01:38:58,088 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_102.eml
2026-05-08 01:38:58,197 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_103.eml
2026-05-08 01:38:58,270 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_104.eml
2026-05-08 01:38:58,387 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_105.eml
2026-05-08 01:38:58,464 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedba

2026-05-08 01:39:05,304 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_164.eml
2026-05-08 01:39:05,368 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_165.eml
2026-05-08 01:39:05,464 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_166.eml
2026-05-08 01:39:05,537 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_167.eml
2026-05-08 01:39:05,643 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_168.eml
2026-05-08 01:39:05,744 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_169.eml
2026-05-08 01:39:05,804 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_17.eml
2026-05-08 01:39:05,915 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_170.eml
2026-05-08 01:39:05,966 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feed

2026-05-08 01:39:10,940 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_229.eml
2026-05-08 01:39:11,015 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_23.eml
2026-05-08 01:39:11,097 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_230.eml
2026-05-08 01:39:11,183 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_231.eml
2026-05-08 01:39:11,252 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_232.eml
2026-05-08 01:39:11,314 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_233.eml
2026-05-08 01:39:11,428 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_234.eml
2026-05-08 01:39:11,508 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_235.eml
2026-05-08 01:39:11,605 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feed

2026-05-08 01:39:17,437 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_294.eml
2026-05-08 01:39:17,536 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_295.eml
2026-05-08 01:39:17,620 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_296.eml
2026-05-08 01:39:17,709 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_297.eml
2026-05-08 01:39:17,782 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_298.eml
2026-05-08 01:39:17,857 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_299.eml
2026-05-08 01:39:17,977 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_3.eml
2026-05-08 01:39:18,044 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_30.eml
2026-05-08 01:39:18,102 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedba

2026-05-08 01:39:23,107 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_359.eml
2026-05-08 01:39:23,188 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_36.eml
2026-05-08 01:39:23,265 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_360.eml
2026-05-08 01:39:23,331 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_361.eml
2026-05-08 01:39:23,401 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_362.eml
2026-05-08 01:39:23,456 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_363.eml
2026-05-08 01:39:23,522 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_364.eml
2026-05-08 01:39:23,621 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_365.eml
2026-05-08 01:39:23,717 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feed

2026-05-08 01:39:28,573 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_44.eml
2026-05-08 01:39:28,657 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_45.eml
2026-05-08 01:39:28,714 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_46.eml
2026-05-08 01:39:28,805 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_47.eml
2026-05-08 01:39:28,858 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_48.eml
2026-05-08 01:39:28,925 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_49.eml
2026-05-08 01:39:28,982 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_5.eml
2026-05-08 01:39:29,080 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_faker/email_feedback_50.eml
2026-05-08 01:39:29,155 - INFO - Lendo arquivo .eml: 2026-05-04-22-32/emails_feedback_fak

In [22]:
dados_emails

[{'remetente': 'cannonjanet@example.net',
  'destinatario': 'xellis@example.com',
  'assunto': 'Feedback sobre estadia #1',
  'data': 'Sun, 08 Sep 2024 23:42:25',
  'corpo': 'A cidade é agradável, mas a gastronomia poderia ter mais opções diferenciadas.\r\n    '},
 {'remetente': 'imack@example.com',
  'destinatario': 'darlene71@example.net',
  'assunto': 'Feedback sobre estadia #10',
  'data': 'Tue, 03 Sep 2024 21:43:10',
  'corpo': 'Foi uma viagem satisfatória, a cidade tem boas opções, mas nada muito surpreendente.\r\n    '},
 {'remetente': 'gonzalesjustin@example.com',
  'destinatario': 'mindydiaz@example.org',
  'assunto': 'Feedback sobre estadia #100',
  'data': 'Sat, 21 Sep 2024 09:37:03',
  'corpo': 'A estadia foi maravilhosa, adoramos os restaurantes e os passeios turísticos!\r\n    '},
 {'remetente': 'jessicapena@example.net',
  'destinatario': 'abrown@example.net',
  'assunto': 'Feedback sobre estadia #101',
  'data': 'Sun, 22 Sep 2024 22:29:12',
  'corpo': 'Sensacional! A ci

In [23]:
# Convertendo a lista para um DataFrame Pandas
try:
    logging.info("Convertendo os dados dos e-mails para DataFrame Pandas.")
    df_emails = pd.DataFrame(dados_emails)
except Exception as e:
    logging.error(f"Erro ao converter os dados dos e-mails para DataFrame. Erro: {e}")
    raise e

2026-05-08 01:41:18,610 - INFO - Convertendo os dados dos e-mails para DataFrame Pandas.


In [24]:
df_emails.head()

,remetente,destinatario,assunto,data,corpo
0,cannonjanet@example.net,xellis@example.com,Feedback sobre estadia #1,"Sun, 08 Sep 2024 23:42:25","A cidade é agradável, mas a gastronomia poderi..."
1,imack@example.com,darlene71@example.net,Feedback sobre estadia #10,"Tue, 03 Sep 2024 21:43:10","Foi uma viagem satisfatória, a cidade tem boas..."
2,gonzalesjustin@example.com,mindydiaz@example.org,Feedback sobre estadia #100,"Sat, 21 Sep 2024 09:37:03","A estadia foi maravilhosa, adoramos os restaur..."
3,jessicapena@example.net,abrown@example.net,Feedback sobre estadia #101,"Sun, 22 Sep 2024 22:29:12","Sensacional! A cidade é linda e segura, os res..."
4,egraham@example.com,martinjackson@example.com,Feedback sobre estadia #102,"Tue, 03 Sep 2024 01:52:14","Tivemos uma experiência incrível, a segurança ..."


In [27]:
# Carregando os dados no BigQuery usando pandas-gbq
try:
    logging.info(f"Carregando os dados no BigQuery: {tabela_bigquery}...")

    # Usando pandas-gbq para carregar os dados
    to_gbq(df_emails, tabela_bigquery, project_id=projeto_bigquery, if_exists='replace')    
    logging.info("Dados carregados com sucesso no BigQuery.")
    
except Exception as e:
    logging.error(f"Erro ao carregar os dados no BigQuery. Erro: {e}")
    raise e

2026-05-08 02:12:48,015 - INFO - Carregando os dados no BigQuery: insight_data.tb_emails_feedback...
421 out of 421 rows loaded.<?, ?it/s]2026-05-08 02:12:54,482 - INFO - 
100%|██████████| 1/1 [00:00<00:00, 856.33it/s]
2026-05-08 02:12:54,483 - INFO - Dados carregados com sucesso no BigQuery.
